In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import forecast_gnarx as fg          # runs the primary pipeline on import

# guard: importing must not have changed the primary result
_fc7 = os.path.join(fg.OUT_DIR, "FC7_gnarx_metrics.csv")
_now = pd.read_csv(_fc7)
assert len(_now) == 224, "FC7 has an unexpected shape after import"
print("\n" + "=" * 74)
print("ADJACENCY SENSITIVITY")
print("=" * 74)

LAKES, n, N = fg.LAKES, fg.n, fg.N
HOR, targets = fg.HORIZONS, fg.targets
SEED = 20260808
N_NULL = 200                     # matches the CCM negative control's power
ABL = {"GNAR": [], "GNARX_wb": ["wb"], "GNARX_wb_dmi": ["wb", "dmi"]}
NULL_ABL = ["GNAR", "GNARX_wb"]  # the two that isolate the network cleanly


def load_adj(path):
    a = pd.read_csv(path, index_col=0)
    a.index = [str(x) for x in a.index]
    a = a.reindex(index=LAKES, columns=LAKES)
    assert a.notna().all().all(), f"{path} does not cover all seven lakes"
    A = a.values.astype(float)
    np.fill_diagonal(A, 0.0)
    return A


def top_k_rows(M, k=3):
    """Keep the k largest off-diagonal entries in each row. This is exactly
    how ccm_network.py built CCM_09_adjacency_top3_*, so a correlation graph
    built this way has identical density to the primary graph."""
    A = np.abs(np.asarray(M, float)).copy()
    np.fill_diagonal(A, 0.0)
    out = np.zeros_like(A)
    for i in range(A.shape[0]):
        idx = np.argsort(A[i])[::-1][:k]
        out[i, idx] = A[i, idx]
    return out


CCM = fg.P("Code Outputs", "CCM Outputs")
A_primary = fg.A
GRAPHS = {
    "ccm_diff_train":  A_primary,
    "ccm_stl_train":   load_adj(os.path.join(CCM, "CCM_09_adjacency_top3_stl_train.csv")),
    "ccm_strict_diff": load_adj(os.path.join(CCM, "CCM_11_adjacency_strict_diff_train.csv")),
    "corr_top3":       top_k_rows(pd.read_csv(
                            fg.P("Code Outputs", "EDA Outputs",
                                 "EDA_06_corr_residual.csv"), index_col=0)
                            .reindex(index=LAKES, columns=LAKES).values, 3),
    "binary_diff":     (A_primary > 0).astype(float),
    "complete":        1.0 - np.eye(n),
}

assert np.array_equal(GRAPHS["ccm_diff_train"], A_primary), "primary graph drifted"
print("\n[1] graphs")
for g, A in GRAPHS.items():
    iso = [LAKES[i].replace("Lake ", "") for i in range(n) if (A[i] > 0).sum() == 0]
    print(f"    {g:17s} edges={int((A>0).sum()):3d}  "
          f"mean w={A[A>0].mean():.3f}" if (A > 0).any() else f"    {g:17s} empty",
          ("  no in-edges: " + ",".join(iso)) if iso else "")

# how much of the primary graph does each alternative reproduce?
pe = (A_primary > 0)
for g, A in GRAPHS.items():
    if g == "ccm_diff_train":
        continue
    e = (A > 0)
    inter = int((pe & e).sum())
    print(f"    {g:17s} shares {inter:2d}/{int(pe.sum())} edges with the primary graph")

# scoring, using the SARIMA denominator from the shared baseline
SAR = {}
for lk in LAKES:
    for h in HOR:
        Pv, Av = [], []
        for o in targets:
            t = o + h - 1
            if t >= N or fg.INTERP[lk].values[t]:
                continue
            Pv.append(fg.preds[("SARIMA", lk, t, h)])
            Av.append(fg.L[lk].values[t])
        SAR[(lk, h)] = round(fg.rmse(Pv, Av), 4)


def score(store, label):
    rows = []
    for lk in LAKES:
        for h in HOR:
            Pv, Av = [], []
            for o in targets:
                t = o + h - 1
                if t >= N or fg.INTERP[lk].values[t]:
                    continue
                Pv.append(store[(label, lk, t, h)])
                Av.append(fg.L[lk].values[t])
            assert len(Pv) == fg.EXPECTED_N[h], (
                f"{label} {lk} h={h}: {len(Pv)} forecasts, "
                f"expected {fg.EXPECTED_N[h]}")
            r = round(fg.rmse(Pv, Av), 4)
            rows.append({"Lake": lk, "Horizon_m": h, "RMSE_m": r,
                         "MAE_m": round(fg.mae(Pv, Av), 4),
                         "n_scored": len(Pv),
                         "skill_vs_SARIMA_%": round(
                             100 * (SAR[(lk, h)] - r) / SAR[(lk, h)], 1)})
    return pd.DataFrame(rows)


def evaluate(A, keys, name, select_order=True):

    if select_order:
        best = None
        for p in fg.P_GRID:
            for s in fg.s_grid(p):
                m = fg.fit_gnarx(keys, p, s, 1, fg.split, A)
                if best is None or m.bic() < best[0]:
                    best = (m.bic(), p, s)
        order = (best[1], best[2])
    else:
        order = fg.CHOSEN[name]
    store = {}
    fg.run(name, keys, refit=False, adj=A, order=order, store=store,
           label="X")
    return score(store, "X"), order



# Graphs

print("\n[2] fitting the six graphs x three ablations")
rows, summary = [], []
for gname, A in GRAPHS.items():
    for mname, keys in ABL.items():
        met, order = evaluate(A, keys, mname)
        met.insert(0, "Graph", gname)
        met.insert(1, "Model", mname)
        met["p"], met["s"] = order[0], str(order[1])
        rows.append(met)
        mean = met.groupby("Horizon_m")["skill_vs_SARIMA_%"].mean()
        summary.append({"Graph": gname, "Model": mname,
                        "p": order[0], "s": str(order[1]),
                        **{f"h{h}": round(mean[h], 1) for h in HOR}})
    print(f"    {gname} done")

SENS = pd.concat(rows, ignore_index=True)

# Regression test
# The primary graph run through this script must reproduce FC7 exactly
_p = SENS[SENS.Graph == "ccm_diff_train"]
for _m in ABL:
    _x = _now[_now.Model == _m][["Lake", "Horizon_m", "RMSE_m",
                                 "skill_vs_SARIMA_%"]]
    _y = _p[_p.Model == _m][["Lake", "Horizon_m", "RMSE_m",
                             "skill_vs_SARIMA_%"]]
    _j = _x.merge(_y, on=["Lake", "Horizon_m"], suffixes=("_fc7", "_fc8"))
    assert len(_j) == 28, f"{_m}: expected 28 rows, got {len(_j)}"
    assert (_j.RMSE_m_fc7 == _j.RMSE_m_fc8).all(), f"{_m}: RMSE drift vs FC7"
    assert (_j["skill_vs_SARIMA_%_fc7"] ==
            _j["skill_vs_SARIMA_%_fc8"]).all(), f"{_m}: skill drift vs FC7"
print("    regression test: primary graph reproduces FC7 exactly "
      "(RMSE and skill, all 84 rows)")

SENS.to_csv(os.path.join(fg.OUT_DIR, "FC8_adjacency_sensitivity.csv"), index=False)
SUM = pd.DataFrame(summary)

# the random-graph null control

print(f"\n[3] null control: {N_NULL} density-matched random graphs")
rng = np.random.default_rng(SEED)
k_edges = int((A_primary > 0).sum())
w_lo, w_hi = A_primary[A_primary > 0].min(), A_primary[A_primary > 0].max()

null_rows = []
for trial in range(N_NULL):
    A = np.zeros((n, n))
    off = [(i, j) for i in range(n) for j in range(n) if i != j]
    pick = rng.choice(len(off), size=k_edges, replace=False)
    for q in pick:
        i, j = off[q]
        A[i, j] = rng.uniform(w_lo, w_hi)
    for mname in NULL_ABL:
        met, _ = evaluate(A, ABL[mname], mname, select_order=False)
        mean = met.groupby("Horizon_m")["skill_vs_SARIMA_%"].mean()
        null_rows.append({"trial": trial, "Model": mname,
                          **{f"h{h}": round(mean[h], 3) for h in HOR}})
    if (trial + 1) % 50 == 0:
        print(f"    {trial+1}/{N_NULL}")

NULL = pd.DataFrame(null_rows)
NULL.to_csv(os.path.join(fg.OUT_DIR, "FC8_null_distribution.csv"), index=False)

# where does each real graph sit in the null distribution?
for h in HOR:
    col = f"h{h}"
    SUM[f"null_pctile_h{h}"] = SUM.apply(
        lambda r: round(100 * (NULL[NULL.Model == r.Model][col] < r[col]).mean(), 1)
        if r.Model in NULL_ABL else np.nan, axis=1)
SUM.to_csv(os.path.join(fg.OUT_DIR, "FC8_sensitivity_summary.csv"), index=False)

# report
print("\n" + "=" * 74)
print("MEAN skill vs SARIMA (%) by graph x model x horizon")
print("=" * 74)
for mname in ABL:
    d = SUM[SUM.Model == mname].set_index("Graph")
    print(f"\n  {mname}")
    print(d[["p", "s", "h1", "h3", "h6", "h12"]].to_string())

print("\n" + "=" * 74)
print(f"NULL CONTROL: {N_NULL} random graphs, density matched ({k_edges} edges)")
print("=" * 74)
for mname in NULL_ABL:
    d = NULL[NULL.Model == mname]
    print(f"\n  {mname} - random-graph distribution of mean skill")
    for h in HOR:
        c = d[f"h{h}"]
        real = SUM[(SUM.Model == mname) &
                   (SUM.Graph == "ccm_diff_train")][f"h{h}"].iloc[0]
        pct = 100 * (c < real).mean()
        print(f"    h={h:<2d} null mean {c.mean():6.2f}  "
              f"[p5 {c.quantile(.05):6.2f}, p95 {c.quantile(.95):6.2f}]   "
              f"CCM graph {real:6.2f}  -> {pct:5.1f}th percentile")

print("\nRange across the six real graphs (max - min mean skill):")
for mname in ABL:
    d = SUM[SUM.Model == mname]
    print(f"    {mname:14s} " + "  ".join(
        f"h={h}: {d[f'h{h}'].max()-d[f'h{h}'].min():.1f}" for h in HOR))

# Figure
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
for ax, mname in zip(axes, ABL):
    d = SUM[SUM.Model == mname].set_index("Graph")[[f"h{h}" for h in HOR]]
    d.columns = [str(h) for h in HOR]
    d.T.plot(marker="o", ax=ax)
    if mname in NULL_ABL:
        nd = NULL[NULL.Model == mname]
        lo = [nd[f"h{h}"].quantile(.05) for h in HOR]
        hi = [nd[f"h{h}"].quantile(.95) for h in HOR]
        ax.fill_between(range(len(HOR)), lo, hi, color="grey", alpha=.25,
                        label="random-graph 5-95%")
    ax.axhline(0, color="k", lw=1)
    ax.set_title(mname, fontweight="bold")
    ax.set_xlabel("horizon (months)")
    ax.set_ylabel("mean skill vs SARIMA (%)")
    ax.legend(fontsize=7)
plt.suptitle("Does the GNARX result depend on which network estimator you "
             "believe?", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(fg.OUT_DIR, "FC8_sensitivity.png"), dpi=200)
plt.close()

print("\nWritten to", fg.OUT_DIR)
for f in ("FC8_adjacency_sensitivity.csv", "FC8_sensitivity_summary.csv",
          "FC8_null_distribution.csv", "FC8_sensitivity.png"):
    print("   ", f)


ADJACENCY SENSITIVITY

[1] graphs
    ccm_diff_train    edges= 20  mean w=0.598 
    ccm_stl_train     edges= 21  mean w=0.521 
    ccm_strict_diff   edges=  9  mean w=0.603   no in-edges: Edward,Kivu,Turkana
    corr_top3         edges= 21  mean w=0.452 
    binary_diff       edges= 20  mean w=1.000 
    complete          edges= 42  mean w=1.000 
    ccm_stl_train     shares 11/20 edges with the primary graph
    ccm_strict_diff   shares  6/20 edges with the primary graph
    corr_top3         shares 11/20 edges with the primary graph
    binary_diff       shares 20/20 edges with the primary graph
    complete          shares 20/20 edges with the primary graph

[2] fitting the six graphs x three ablations
    ccm_diff_train done
    ccm_stl_train done
    ccm_strict_diff done
    corr_top3 done
    binary_diff done
    complete done
    regression test: primary graph reproduces FC7 exactly (RMSE and skill, all 84 rows)

[3] null control: 200 density-matched random graphs
    50/200
 